# ZETTY RBA — Google Colab 전용 학습
Mac 학습은 중단되어 있습니다. **Google 호스팅 런타임**을 연결하세요. 로컬 런타임을 선택하지 마세요. Colab 메뉴 **런타임 → 런타임 유형 변경 → 제공되는 GPU**를 선택하세요. 기본 profile은 GPU XGBoost 4개 + CPU Isolation Forest 기준선 2개입니다. GPU 미할당 시 시작 전에 중단합니다. 무료 GPU 종류·연속 실행 시간은 보장되지 않습니다.

완료된 Online Shop/RBD 179개 실험은 다시 실행하지 않습니다. `resume`은 동일 profile/Python/의존성의 중단된 연구만 재개합니다. 기존 CPU 연구는 PROFILE=cpu-full로 선택해야 하며 GPU 연구와 섞지 않습니다. `fresh`는 RBA만 새로 전처리·학습합니다. `prepared`는 이전 전처리 배열을 재사용해 새 GPU 연구를 시작합니다. 무료 Colab의 RAM/연속 실행 시간은 보장되지 않습니다.


## 1. 원격 runtime / Python 확인
Python 3.12/3.13만 지원합니다. resume은 checkpoint와 같은 Python minor가 필요합니다. Drive 승인은 이후 셀에서 수행합니다.


In [ ]:
import os, sys, platform
assert platform.system() == 'Linux' and os.environ.get('COLAB_RELEASE_TAG'), 'Google-hosted Colab required; do not use Mac local runtime'
assert sys.version_info[:2] in ((3, 12), (3, 13)), f'Python 3.12 or 3.13 required, got {sys.version}'
print(platform.platform(), sys.version)

PROFILE = 'colab-gpu-v1'  # legacy CPU study: 'cpu-full'
if PROFILE == 'colab-gpu-v1':
    import subprocess
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version', '--format=csv'], check=True)


## 2. 코드 및 고정 라이브러리 설치
설치 뒤 NumPy/scikit-learn을 이미 import한 세션이라면 런타임을 다시 시작하고 1번부터 실행하세요.


In [ ]:
import subprocess
from pathlib import Path
REPO = Path('/content/zetty-uba')
REF = 'feature/public-data-training-star'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', REF, '--single-branch', 'https://github.com/jjyj0203/uba-analyzer.git', str(REPO)], check=True)
# Existing clones must not silently run stale code. Preserve local edits.
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REF], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
remote = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
assert head == remote, 'Old checkout: use a new Colab runtime and rerun from step 1'
assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'Modified checkout: use a new runtime'
print('Code commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', '-r', str(REPO/('requirements-colab-gpu.lock' if PROFILE == 'colab-gpu-v1' else 'requirements-ml.lock'))], check=True)

# Check compiled packages in a clean process before downloading/training.
subprocess.run([sys.executable, '-c', 'import numpy, scipy, sklearn, joblib, threadpoolctl, pyarrow; print("ML imports OK")'], check=True)

if PROFILE == 'colab-gpu-v1':
    gpu_env = dict(os.environ, PYTHONPATH=str(REPO/'src'))
    subprocess.run([sys.executable, '-m', 'zetty_uba.study.gpu'], env=gpu_env, check=True)


## 3. Drive 연결 및 모드 선택
`resume`: Mac의 `ml-runs/20260927-full`에서 **prepared/rba와 runs/rba**를 같은 구조로 Drive의 `zetty/20260927-full`에 업로드합니다. 약 2.7 GB의 배열과 모델/분할 파일이 필요합니다. `source_snapshot`을 빠뜨리지 마세요.

`fresh`: 업로드 없이 공식 원자료를 Colab에서 받습니다. 기존 모델을 덮어쓰지 않는 새 GPU 비교 연구입니다. `prepared`는 준비된 배열만 재사용하며 기존 Python 버전과 달라도 모델을 읽지 않으므로 새 학습에 사용할 수 있습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MODE = 'fresh'  # fresh | prepared (reuse arrays only) | resume (same profile/Python)
CHECKPOINT = Path('/content/drive/MyDrive/zetty/20260927-full')
from datetime import datetime, timezone
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUTPUT = Path('/content') / ('zetty-rba-' + RUN_ID)
BACKUP = Path('/content/drive/MyDrive/zetty/colab-runs') / RUN_ID
print('Mode:', MODE, '\nLocal VM:', OUTPUT, '\nDrive backup:', BACKUP)


## 4. 지금까지 학습한 모델 확인 — 학습 실행 없음
EVALUATED는 최종 test까지 완료, VALIDATED는 학습·보정·validation만 완료입니다. RBA의 최종 test는 아직 없습니다.


In [ ]:
import json
if MODE == 'resume':
    saved = json.loads((CHECKPOINT/'runs/rba/study.json').read_text())
    saved_minor = tuple(int(x) for x in saved['environment']['python'].split('.')[:2])
    assert saved.get('profile', 'cpu-full') == PROFILE, 'Profile mismatch: use MODE=prepared for a new GPU study'
    assert saved_minor == sys.version_info[:2], 'Checkpoint Python mismatch: use matching runtime or MODE=fresh'
    assert (CHECKPOINT/'prepared/rba/dataset.json').is_file(), 'Upload prepared/rba first'
    assert (CHECKPOINT/'runs/rba/source_snapshot/study/runner.py').is_file(), 'Upload runs/rba including source_snapshot'
    subprocess.run([sys.executable, str(REPO/'scripts/ml/inspect_study.py'), str(CHECKPOINT/'runs/rba')], check=True)
elif MODE == 'prepared':
    assert (CHECKPOINT/'prepared/rba/dataset.json').is_file(), 'Upload prepared/rba first'
    print('Reusing completed arrays only; new models and study. No old model loading.')
else:
    print('Fresh mode: original Mac results remain unchanged. See docs/ml/07-full-data-study/comparison.csv')


## 5. Colab에서 학습 실행
Drive 자료를 VM 로컬 디스크로 복사하고 학습합니다. 완료 artifact를 30초 간격으로 백업합니다. 재개 시 원본 Mac run을 수정하지 않고 새 출력 디렉터리를 사용합니다. 진행 중인 모델 fit이 끊기면 해당 모델은 처음부터 다시 수행합니다. 완료된 후보는 동일 profile resume에서 재사용합니다.


In [ ]:
command = [sys.executable, str(REPO/'scripts/ml/colab_rba.py'), '--mode', MODE, '--profile', PROFILE, '--output', str(OUTPUT), '--backup', str(BACKUP)]
if MODE in ('resume', 'prepared'):
    command += ['--checkpoint', str(CHECKPOINT)]
subprocess.run(command, check=True)


## 6. 완료 상태·최종 성능 확인
최종 test를 보고 모델을 재선택하지 않습니다. selection은 validation만으로 고정됩니다. COMPLETED일 때만 추론 환경에 반영합니다.


In [ ]:
import json
manifest = json.loads((OUTPUT/'runs/rba/study.json').read_text())
print('Status:', manifest['status'])
print('Selection:', manifest.get('selection'))
subprocess.run([sys.executable, str(REPO/'scripts/ml/inspect_study.py'), str(OUTPUT/'runs/rba')], check=True)
assert manifest['status'] == 'COMPLETED', 'Not ready for inference deployment'


## 7. 결과 내려받기
ZIP에는 모델, threshold/feature manifest, source snapshot, split과 cloud 실행 기록을 담습니다. Mac에서 새 버전 디렉터리에 보관하세요. XGBoost 결과는 requirements-xgb-inference.lock 의존성이 있는 RBA 피처 추론 환경에서 사용합니다. 기존 HTTP Compose 모델을 이 RBA 모델로 교체하지 마세요. 기존 RBD 모델을 덮어쓰지 않습니다.


In [ ]:
import zipfile
archive = OUTPUT/'zetty-rba-completed.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED) as z:
    for path in (OUTPUT/'runs/rba').rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and path.suffix != '.tmp':
            z.write(path, path.relative_to(OUTPUT/'runs'))
from google.colab import files
files.download(str(archive))
print('Persistent Drive copy:', BACKUP/'runs/rba')


## 중단·재개
Colab이 종료되면 Drive BACKUP을 다음 실행의 CHECKPOINT로 지정하고 `resume`으로 실행합니다. 원본 배열 백업이 끝나기 전 종료되면 최초 Mac checkpoint 또는 `fresh`가 필요합니다. 작업이 끝나면 **런타임 연결 해제 및 삭제**로 자원을 반환하세요.

이 notebook JSON/구문은 로컬에서 확인했지만 Google 계정 로그인·Drive 업로드·원격 학습은 사용자가 실행해야 합니다. [Colab FAQ](https://research.google.com/colaboratory/faq.html)
